In [16]:
# Dataset Exploration

## Daily Activities Wearable Dataset for Cardiorespiratory Fitness Estimation

#This notebook explores the dataset before any preprocessing or model development.

### Objectives

#- Understand the dataset structure
#- Explore participant metadata
#- Inspect IMU sensor data
#- Inspect biomarker data
#- Identify input features and labels
#- Verify data quality before preprocessing

In [1]:
# Cell 1 - Locate Dataset and Discover Subject Folders

from pathlib import Path
import os
import re

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# Use an environment variable on Narval when available.
# Otherwise, use the local relative path.
DATASET_PATH = Path(
    os.environ.get(
        "IMU_DATASET_PATH",
        "../datasets/DatasetIMUandBIOMARKERS"
    )
).resolve()


print("=" * 65)
print("DATASET LOCATION")
print("=" * 65)

print("Dataset path   :", DATASET_PATH)
print("Dataset exists :", DATASET_PATH.exists())


if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset directory was not found:\n{DATASET_PATH}\n\n"
        "Set the IMU_DATASET_PATH environment variable or "
        "update DATASET_PATH."
    )


# Select folders that exactly follow Subject + number,
# such as Subject01 or Subject67.
subject_pattern = re.compile(r"^Subject\d+$")

subjects = sorted(
    [
        item
        for item in DATASET_PATH.iterdir()
        if item.is_dir()
        and subject_pattern.match(item.name)
    ],
    key=lambda path: int(
        re.search(r"\d+", path.name).group()
    )
)


print("\nNumber of subject folders:", len(subjects))

print("\nFirst five subjects:")
for subject in subjects[:5]:
    print(" -", subject.name)

print("\nLast five subjects:")
for subject in subjects[-5:]:
    print(" -", subject.name)

DATASET LOCATION
Dataset path   : C:\PrivDiffuser_Narval\datasets\DatasetIMUandBIOMARKERS
Dataset exists : True

Number of subject folders: 67

First five subjects:
 - Subject01
 - Subject02
 - Subject03
 - Subject04
 - Subject05

Last five subjects:
 - Subject63
 - Subject64
 - Subject65
 - Subject66
 - Subject67


In [2]:
# Load participant metadata

metadata_path = DATASET_PATH/"SubjectsInfo.xlsx"

metadata = pd.read_excel(metadata_path)

print("Metadata loaded successfully!")

print(f"Shape: {metadata.shape}")

# Display the first five rows
metadata.head()

# Display column names
print("Metadata Columns:\n")

for i, column in enumerate(metadata.columns):
    print(f"{i+1}. {column}")

# Basic information
metadata.info()

Metadata loaded successfully!
Shape: (60, 10)
Metadata Columns:

1. Participant ID
2. Gender
3. Age
4. Height (m)
5. Weight (kg)
6. Fat %
7. BMI
8. SpO2_baseline(%)
9. HR_baseline(bpm)
10. HR step test(bpm)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Participant ID     60 non-null     int64  
 1   Gender             60 non-null     object 
 2   Age                60 non-null     int64  
 3   Height (m)         60 non-null     float64
 4   Weight (kg)        60 non-null     float64
 5   Fat %              60 non-null     float64
 6   BMI                60 non-null     float64
 7   SpO2_baseline(%)   60 non-null     int64  
 8   HR_baseline(bpm)   60 non-null     int64  
 9   HR step test(bpm)  60 non-null     int64  
dtypes: float64(4), int64(5), object(1)
memory usage: 4.8+ KB


In [3]:
# Load one participant's IMU data

imu_path = DATASET_PATH / "Subject01" / "IMUSubject01.csv"

imu = pd.read_csv(imu_path)

print("IMU data loaded successfully!")

print(f"Shape: {imu.shape}")
imu.head()
print("IMU Columns:\n")

for i, column in enumerate(imu.columns):
    print(f"{i+1}. {column}")

IMU data loaded successfully!
Shape: (110740, 32)
IMU Columns:

1. epoch
2. timestamp_unified
3. q_w_chest
4. q_x_chest
5. q_y_chest
6. q_z_chest
7. q_w_left_hand
8. q_x_left_hand
9. q_y_left_hand
10. q_z_left_hand
11. q_w_right_knee
12. q_x_right_knee
13. q_y_right_knee
14. q_z_right_knee
15. a_x_chest
16. a_y_chest
17. a_z_chest
18. g_x_chest
19. g_y_chest
20. g_z_chest
21. a_x_left_knee
22. a_y_left_knee
23. a_z_left_knee
24. g_x_left_knee
25. g_y_left_knee
26. g_z_left_knee
27. a_x_right_hand
28. a_y_right_hand
29. a_z_right_hand
30. g_x_right_hand
31. g_y_right_hand
32. g_z_right_hand


In [4]:
# Display the first five IMU samples
imu.head()
# Basic information about the IMU dataset
print("IMU Dataset Information:\n")
imu.info()


IMU Dataset Information:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110740 entries, 0 to 110739
Data columns (total 32 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   epoch              110740 non-null  float64
 1   timestamp_unified  110740 non-null  object 
 2   q_w_chest          110740 non-null  float64
 3   q_x_chest          110740 non-null  float64
 4   q_y_chest          110740 non-null  float64
 5   q_z_chest          110740 non-null  float64
 6   q_w_left_hand      110740 non-null  float64
 7   q_x_left_hand      110740 non-null  float64
 8   q_y_left_hand      110740 non-null  float64
 9   q_z_left_hand      110740 non-null  float64
 10  q_w_right_knee     110740 non-null  float64
 11  q_x_right_knee     110740 non-null  float64
 12  q_y_right_knee     110740 non-null  float64
 13  q_z_right_knee     110740 non-null  float64
 14  a_x_chest          110740 non-null  float64
 15  a_y_chest          110740

In [5]:
# Check for missing values
print("Missing values in each column:\n")
print(imu.isnull().sum())

#Summary statistics
imu.describe()

# Data types of each column
print(imu.dtypes)

Missing values in each column:

epoch                0
timestamp_unified    0
q_w_chest            0
q_x_chest            0
q_y_chest            0
q_z_chest            0
q_w_left_hand        0
q_x_left_hand        0
q_y_left_hand        0
q_z_left_hand        0
q_w_right_knee       0
q_x_right_knee       0
q_y_right_knee       0
q_z_right_knee       0
a_x_chest            0
a_y_chest            0
a_z_chest            0
g_x_chest            0
g_y_chest            0
g_z_chest            0
a_x_left_knee        0
a_y_left_knee        0
a_z_left_knee        0
g_x_left_knee        0
g_y_left_knee        0
g_z_left_knee        0
a_x_right_hand       0
a_y_right_hand       0
a_z_right_hand       0
g_x_right_hand       0
g_y_right_hand       0
g_z_right_hand       0
dtype: int64
epoch                float64
timestamp_unified     object
q_w_chest            float64
q_x_chest            float64
q_y_chest            float64
q_z_chest            float64
q_w_left_hand        float64
q_x_left_hand   

In [6]:
# Count sensor modalities

accelerometer_cols = [c for c in imu.columns if c.startswith("a_")]
gyroscope_cols = [c for c in imu.columns if c.startswith("g_")]
quaternion_cols = [c for c in imu.columns if c.startswith("q_")]

print("Accelerometer channels :", len(accelerometer_cols))
print(accelerometer_cols)

print()

print("Gyroscope channels :", len(gyroscope_cols))
print(gyroscope_cols)

print()

print("Quaternion channels :", len(quaternion_cols))
print(quaternion_cols)

Accelerometer channels : 9
['a_x_chest', 'a_y_chest', 'a_z_chest', 'a_x_left_knee', 'a_y_left_knee', 'a_z_left_knee', 'a_x_right_hand', 'a_y_right_hand', 'a_z_right_hand']

Gyroscope channels : 9
['g_x_chest', 'g_y_chest', 'g_z_chest', 'g_x_left_knee', 'g_y_left_knee', 'g_z_left_knee', 'g_x_right_hand', 'g_y_right_hand', 'g_z_right_hand']

Quaternion channels : 12
['q_w_chest', 'q_x_chest', 'q_y_chest', 'q_z_chest', 'q_w_left_hand', 'q_x_left_hand', 'q_y_left_hand', 'q_z_left_hand', 'q_w_right_knee', 'q_x_right_knee', 'q_y_right_knee', 'q_z_right_knee']


In [7]:
# Explore all IMU files

subject_summary = []

for subject in subjects:
    
    imu_file = subject / f"IMUSubject{subject.name[-2:]}.csv"

    if imu_file.exists():
        df = pd.read_csv(imu_file)

        subject_summary.append({
            "Subject": subject.name,
            "Rows": df.shape[0],
            "Columns": df.shape[1]
        })

summary = pd.DataFrame(subject_summary)

summary.head()
summary.describe()
print(summary)

      Subject    Rows  Columns
0   Subject01  110740       32
1   Subject02  110563       32
2   Subject03   98902       32
3   Subject04  106672       32
4   Subject05  105411       33
..        ...     ...      ...
62  Subject63  109860       33
63  Subject64  111050       33
64  Subject65  107810       33
65  Subject66  102241       33
66  Subject67  108030       33

[67 rows x 3 columns]


In [8]:
# Compare Dataset Shape and Column Names Across Subjects

subject01 = pd.read_csv(
    DATASET_PATH / "Subject01" / "IMUSubject01.csv"
)

subject05 = pd.read_csv(
    DATASET_PATH / "Subject05" / "IMUSubject05.csv"
)

print("=" * 80)
print("SUBJECT01 DATASET INFORMATION")
print("=" * 80)

print("Shape:", subject01.shape)
print("Number of rows:", subject01.shape[0])
print("Number of columns:", subject01.shape[1])
print("Columns:")
print(subject01.columns.tolist())


print("\n" + "=" * 80)
print("SUBJECT05 DATASET INFORMATION")
print("=" * 80)

print("Shape:", subject05.shape)
print("Number of rows:", subject05.shape[0])
print("Number of columns:", subject05.shape[1])
print("Columns:")
print(subject05.columns.tolist())


print("\n" + "=" * 80)
print("COMPARISON")
print("=" * 80)

same_shape = subject01.shape == subject05.shape
same_column_count = (
    subject01.shape[1] == subject05.shape[1]
)
same_columns = (
    subject01.columns.tolist()
    == subject05.columns.tolist()
)

print("Same complete shape:", same_shape)
print("Same number of columns:", same_column_count)
print("Same column names and order:", same_columns)

print(
    "Difference in number of rows:",
    abs(subject01.shape[0] - subject05.shape[0])
)

SUBJECT01 DATASET INFORMATION
Shape: (110740, 32)
Number of rows: 110740
Number of columns: 32
Columns:
['epoch', 'timestamp_unified', 'q_w_chest', 'q_x_chest', 'q_y_chest', 'q_z_chest', 'q_w_left_hand', 'q_x_left_hand', 'q_y_left_hand', 'q_z_left_hand', 'q_w_right_knee', 'q_x_right_knee', 'q_y_right_knee', 'q_z_right_knee', 'a_x_chest', 'a_y_chest', 'a_z_chest', 'g_x_chest', 'g_y_chest', 'g_z_chest', 'a_x_left_knee', 'a_y_left_knee', 'a_z_left_knee', 'g_x_left_knee', 'g_y_left_knee', 'g_z_left_knee', 'a_x_right_hand', 'a_y_right_hand', 'a_z_right_hand', 'g_x_right_hand', 'g_y_right_hand', 'g_z_right_hand']

SUBJECT05 DATASET INFORMATION
Shape: (105411, 33)
Number of rows: 105411
Number of columns: 33
Columns:
['timestamp', 'time', 'q_w_chest', 'q_x_chest', 'q_y_chest', 'q_z_chest', 'q_w_left_hand', 'q_x_left_hand', 'q_y_left_hand', 'q_z_left_hand', 'q_w_right_knee', 'q_x_right_knee', 'q_y_right_knee', 'q_z_right_knee', 'a_x_chest', 'a_y_chest', 'a_z_chest', 'g_x_chest', 'g_y_chest', '